In [ ]:
!pip install yt-dlp

## Step 2: Setup and Configuration
Run this cell to set up the downloader functions.

In [ ]:
import yt_dlp
import os

# Check if running in Colab
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Create downloads folder
DOWNLOAD_FOLDER = "youtube_downloads"
if not os.path.exists(DOWNLOAD_FOLDER):
    os.makedirs(DOWNLOAD_FOLDER)

print("✅ Setup complete!")
print(f"📂 Downloads will be saved to: {DOWNLOAD_FOLDER}/")

## Step 3: Configure Your Download (⚡ ONLY PLACE TO EDIT!)

**This is the ONLY place you need to configure!** All other cells will use these settings.

Just edit the values below and run this cell.

In [ ]:
import json

# ==========================================
# ⚡ EDIT THIS CONFIG - ONLY PLACE TO CHANGE!
# ==========================================

CONFIG = {
    # Paste your YouTube URL here (video or playlist)
    # For multiple videos, use a list: ["url1", "url2", "url3"]
    "url": "https://www.youtube.com/watch?v=wRNz3FvTKmU",
    
    # Is this a playlist? (True/False)
    "is_playlist": False,
    
    # Download type: "video", "audio", or "both"
    # - "video": Download video file
    # - "audio": Download MP3 audio only
    # - "both": Download both video and audio separately
    "download_type": "video",
    
    # Video quality (only applies if download_type is "video" or "both")
    # Options: "best", "1080", "720", "480", "360", "240", "144"
    "quality": "best",
    
    # Audio quality (only applies if download_type is "audio" or "both")
    # Options: "128", "192", "256", "320" (kbps)
    "audio_quality": "192"
}

# ==========================================
# Display Configuration
# ==========================================

print("📋 Download Configuration:")
print("=" * 60)
print(json.dumps(CONFIG, indent=2))
print("=" * 60)
print("\n✅ Configuration saved!")
print("💡 TIP: All cells below will use this CONFIG automatically.")
print("   Just run Step 4 or any advanced option to download!")

## Step 4: Start Download

Run this cell to download based on your configuration above.

In [ ]:
def download_with_config(config):
    """Download based on the configuration settings."""
    
    url = config["url"]
    is_playlist = config["is_playlist"]
    download_type = config["download_type"]
    quality = config["quality"]
    audio_quality = config["audio_quality"]
    
    print(f"\n{'='*60}")
    print(f"🎯 Download Type: {download_type.upper()}")
    print(f"📺 URL: {url}")
    print(f"📁 Is Playlist: {'Yes' if is_playlist else 'No'}")
    if download_type in ["video", "both"]:
        print(f"🎬 Video Quality: {quality}p" if quality != "best" else "🎬 Video Quality: Best Available")
    if download_type in ["audio", "both"]:
        print(f"🎵 Audio Quality: {audio_quality} kbps")
    print(f"{'='*60}\n")
    
    downloaded_files = []
    
    # Download Video
    if download_type in ["video", "both"]:
        print("🔄 Downloading VIDEO...\n")
        
        if quality == "best":
            format_str = 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best'
        else:
            format_str = f'bestvideo[height<={quality}][ext=mp4]+bestaudio[ext=m4a]/best[height<={quality}]'
        
        if is_playlist:
            outtmpl = f'{DOWNLOAD_FOLDER}/%(playlist)s/%(playlist_index)s - %(title)s.%(ext)s'
        else:
            outtmpl = f'{DOWNLOAD_FOLDER}/%(title)s.%(ext)s'
        
        ydl_opts = {
            'format': format_str,
            'outtmpl': outtmpl,
            'merge_output_format': 'mp4',
        }
        
        try:
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                info = ydl.extract_info(url, download=True)
                
                if is_playlist:
                    print(f"\n✅ Playlist VIDEO download complete!")
                    print(f"📁 Total videos: {len(info.get('entries', []))}")
                    downloaded_files.append(f"Playlist: {info.get('title', 'Unknown')} ({len(info.get('entries', []))} videos)")
                else:
                    filename = ydl.prepare_filename(info)
                    print(f"\n✅ VIDEO download complete!")
                    print(f"📄 File: {filename}")
                    downloaded_files.append(filename)
        except Exception as e:
            print(f"❌ Error downloading video: {e}")
    
    # Download Audio
    if download_type in ["audio", "both"]:
        print("\n" + "="*60)
        print("🔄 Downloading AUDIO (MP3)...\n")
        
        if is_playlist:
            outtmpl = f'{DOWNLOAD_FOLDER}/%(playlist)s/%(playlist_index)s - %(title)s.%(ext)s'
        else:
            outtmpl = f'{DOWNLOAD_FOLDER}/%(title)s.%(ext)s'
        
        ydl_opts = {
            'format': 'bestaudio/best',
            'outtmpl': outtmpl,
            'postprocessors': [{
                'key': 'FFmpegExtractAudio',
                'preferredcodec': 'mp3',
                'preferredquality': audio_quality,
            }],
        }
        
        try:
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                info = ydl.extract_info(url, download=True)
                
                if is_playlist:
                    print(f"\n✅ Playlist AUDIO download complete!")
                    print(f"📁 Total audio files: {len(info.get('entries', []))}")
                    downloaded_files.append(f"Audio Playlist: {info.get('title', 'Unknown')} ({len(info.get('entries', []))} files)")
                else:
                    base_filename = ydl.prepare_filename(info)
                    mp3_filename = base_filename.rsplit('.', 1)[0] + '.mp3'
                    print(f"\n✅ AUDIO download complete!")
                    print(f"📄 File: {mp3_filename}")
                    downloaded_files.append(mp3_filename)
        except Exception as e:
            print(f"❌ Error downloading audio: {e}")
    
    # Final Summary
    print("\n" + "="*60)
    print("🎉 DOWNLOAD COMPLETE!")
    print("="*60)
    print(f"\n📦 Downloaded {len(downloaded_files)} item(s):")
    for idx, file in enumerate(downloaded_files, 1):
        print(f"  {idx}. {file}")
    print(f"\n📂 Location: {DOWNLOAD_FOLDER}/")
    print("="*60)

# Run the download
try:
    download_with_config(CONFIG)
except NameError:
    print("❌ Error: CONFIG not found!")
    print("Please run the configuration cell (Step 3) first!")
except Exception as e:
    print(f"❌ Unexpected error: {e}")

---

## Alternative Download Options (Advanced)

The cells below are for advanced users who want more control. If you used Steps 3-4 above, you can skip these.

### Option A: Download Best Quality Video (MP4)

In [ ]:
# Uses URL from CONFIG (Step 3)
print(f"🔗 Using URL: {CONFIG['url']}\n")

ydl_opts = {
    'format': 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best',
    'outtmpl': f'{DOWNLOAD_FOLDER}/%(title)s.%(ext)s',
    'merge_output_format': 'mp4',
}

print("🔄 Starting download...\n")

try:
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(CONFIG['url'], download=True)
        filename = ydl.prepare_filename(info)
        print(f"\n✅ Download complete!")
        print(f"📄 File: {filename}")
except Exception as e:
    print(f"❌ Error: {e}")

### Option B: Download Specific Quality (e.g., 720p)

In [ ]:
# Uses URL and quality from CONFIG (Step 3)
QUALITY = CONFIG['quality']
print(f"🔗 Using URL: {CONFIG['url']}")
print(f"🎬 Quality: {QUALITY}\n")

ydl_opts = {
    'format': f'bestvideo[height<={QUALITY}][ext=mp4]+bestaudio[ext=m4a]/best[height<={QUALITY}]' if QUALITY != 'best' else 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best',
    'outtmpl': f'{DOWNLOAD_FOLDER}/%(title)s_{QUALITY}p.%(ext)s',
    'merge_output_format': 'mp4',
}

print(f"🔄 Downloading {QUALITY}p video...\n")

try:
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(CONFIG['url'], download=True)
        filename = ydl.prepare_filename(info)
        print(f"\n✅ Download complete!")
        print(f"📄 File: {filename}")
except Exception as e:
    print(f"❌ Error: {e}")

### Option C: Download Audio Only (MP3)

In [ ]:
# Uses URL and audio quality from CONFIG (Step 3)
print(f"🔗 Using URL: {CONFIG['url']}")
print(f"🎵 Audio Quality: {CONFIG['audio_quality']} kbps\n")

ydl_opts = {
    'format': 'bestaudio/best',
    'outtmpl': f'{DOWNLOAD_FOLDER}/%(title)s.%(ext)s',
    'postprocessors': [{
        'key': 'FFmpegExtractAudio',
        'preferredcodec': 'mp3',
        'preferredquality': CONFIG['audio_quality'],
    }],
}

print("🔄 Downloading audio (MP3)...\n")

try:
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(CONFIG['url'], download=True)
        # Get the final filename after post-processing
        base_filename = ydl.prepare_filename(info)
        mp3_filename = base_filename.rsplit('.', 1)[0] + '.mp3'
        print(f"\n✅ Download complete!")
        print(f"📄 File: {mp3_filename}")
except Exception as e:
    print(f"❌ Error: {e}")

### Option D: Download Multiple Videos (Batch Download)

```

**Note:** To use batch download, edit CONFIG in Step 3 and set:"url": ["url1", "url2", "url3"]  # Put multiple URLs in a list
```python

In [ ]:
# Uses URL(s) from CONFIG (Step 3)
# If CONFIG['url'] is a list, download all; if string, download single
url_config = CONFIG['url']
VIDEO_URLS = url_config if isinstance(url_config, list) else [url_config]

ydl_opts = {
    'format': 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best',
    'outtmpl': f'{DOWNLOAD_FOLDER}/%(title)s.%(ext)s',
    'merge_output_format': 'mp4',
}

print(f"🔄 Downloading {len(VIDEO_URLS)} videos...\n")

for idx, url in enumerate(VIDEO_URLS, 1):
    try:
        print(f"\n[{idx}/{len(VIDEO_URLS)}] Downloading: {url}")
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(url, download=True)
            print(f"✅ Complete: {info['title']}")
    except Exception as e:
        print(f"❌ Error downloading {url}: {e}")

print(f"\n🎉 All downloads complete!")

### Option E: Download Entire Playlist

In [ ]:
# Uses URL from CONFIG (Step 3)
# Make sure to set is_playlist=True in CONFIG
print(f"🔗 Using URL: {CONFIG['url']}")
print(f"📁 Is Playlist: {CONFIG.get('is_playlist', False)}\n")

ydl_opts = {
    'format': 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best',
    'outtmpl': f'{DOWNLOAD_FOLDER}/%(playlist)s/%(playlist_index)s - %(title)s.%(ext)s',
    'merge_output_format': 'mp4',
}

print("🔄 Downloading playlist...\n")

try:
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(CONFIG['url'], download=True)
        print(f"\n✅ Playlist download complete!")
        print(f"📁 Total videos: {len(info['entries'])}")

except Exception as e:    print(f"❌ Error: {e}")

---

## Step 4: View Downloaded Files

In [ ]:
import os

print("📂 Downloaded Files:\n")
print("=" * 60)

total_size = 0
file_count = 0

for root, dirs, files in os.walk(DOWNLOAD_FOLDER):
    for file in files:
        filepath = os.path.join(root, file)
        size = os.path.getsize(filepath)
        total_size += size
        file_count += 1
        
        # Display relative path
        rel_path = os.path.relpath(filepath, DOWNLOAD_FOLDER)
        size_mb = size / (1024 * 1024)
        print(f"📄 {rel_path}")
        print(f"   Size: {size_mb:.2f} MB\n")

print("=" * 60)
print(f"\n📊 Total: {file_count} files, {total_size / (1024 * 1024):.2f} MB")

---

## Step 5: Get Video Information (Without Downloading)

In [ ]:
# Uses URL from CONFIG (Step 3)
print(f"🔗 Using URL: {CONFIG['url']}\n")

ydl_opts = {
    'quiet': True,
    'no_warnings': True,
}

print("🔍 Fetching video information...\n")

try:
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(CONFIG['url'], download=False)
        
        print("📺 Video Information:")
        print("=" * 60)
        print(f"Title: {info.get('title', 'N/A')}")
        print(f"Channel: {info.get('uploader', 'N/A')}")
        print(f"Duration: {info.get('duration', 0) // 60}:{info.get('duration', 0) % 60:02d} minutes")
        print(f"Views: {info.get('view_count', 0):,}")
        print(f"Upload Date: {info.get('upload_date', 'N/A')}")
        print(f"Description: {info.get('description', 'N/A')[:200]}...")
        print("\n📊 Available Formats:")
        print("-" * 60)
        
        # Show available quality options
        formats_seen = set()
        for f in info.get('formats', []):
            if f.get('height'):
                quality = f"{f.get('height')}p"
                ext = f.get('ext', 'unknown')
                format_key = f"{quality}_{ext}"
                if format_key not in formats_seen:
                    formats_seen.add(format_key)
                    filesize = f.get('filesize', 0)
                    if filesize:
                        size_mb = filesize / (1024 * 1024)
                        print(f"  • {quality} ({ext}) - ~{size_mb:.1f} MB")
                    else:
                        print(f"  • {quality} ({ext})")
        
except Exception as e:
    print(f"❌ Error: {e}")

---

## Step 5: View Downloaded Files

Check what files have been downloaded.

In [ ]:
import os

print("📂 Downloaded Files:\n")
print("=" * 70)

total_size = 0
file_count = 0

for root, dirs, files in os.walk(DOWNLOAD_FOLDER):
    for file in files:
        filepath = os.path.join(root, file)
        size = os.path.getsize(filepath)
        total_size += size
        file_count += 1
        
        # Display relative path
        rel_path = os.path.relpath(filepath, DOWNLOAD_FOLDER)
        size_mb = size / (1024 * 1024)
        print(f"📄 {rel_path}")
        print(f"   Size: {size_mb:.2f} MB\n")

print("=" * 70)
print(f"\n📊 Total: {file_count} files, {total_size / (1024 * 1024):.2f} MB")

---

## 📁 For VS Code Users: Find Your Files Manually

If you prefer to access files directly without download links:

1. Look for the `youtube_downloads` folder in your VS Code file explorer (left sidebar)
2. Right-click any file → "Reveal in File Explorer" (Windows) or "Reveal in Finder" (Mac)
3. Your files are there!

**Folder location:** Same directory as this notebook → `youtube_downloads/`

---

## Step 6: Download Files to Your Computer 💾

### ✅ BEST METHOD for VS Code Users!

This generates **clickable download links** that work perfectly in VS Code. No popups, no blocked downloads!

**How to use:**
1. Run this cell
2. Click the green download buttons that appear
3. Files will download directly to your computer

**Note:** Files larger than 100MB will be skipped (too large to encode). For those, check your `youtube_downloads` folder manually.

In [ ]:
import base64
import os
from IPython.display import HTML, display

print("🔗 Scanning files...\n")

# First, scan all files
all_files = []
for root, dirs, files in os.walk(DOWNLOAD_FOLDER):
    for file in files:
        filepath = os.path.join(root, file)
        size = os.path.getsize(filepath)
        all_files.append((filepath, file, size))

if not all_files:
    print("❌ No files found in the download folder!")
    print("Please download some videos first (run Step 4).")
else:
    print(f"📊 Found {len(all_files)} file(s)")
    print(f"⚠️  Note: Encoding large files (>30MB) may take time. Please wait...\n")
    
    html_output = '<div style="background: #f5f5f5; padding: 20px; border-radius: 10px;">'
    html_output += '<h2 style="color: #1976d2; margin-top: 0;">📥 Your Downloaded Files</h2>'
    
    file_found = False
    
    for idx, (filepath, file, size) in enumerate(all_files, 1):
        size_mb = size / (1024 * 1024)
        
        print(f"[{idx}/{len(all_files)}] Processing: {file} ({size_mb:.1f} MB)...", end=" ")
        
        # Check file size before encoding (skip if too large)
        if size > 50 * 1024 * 1024:  # Skip files larger than 50MB (encoding takes too long)
            html_output += f'''
            <div style="background: #fff3cd; padding: 15px; margin: 10px 0; border-radius: 5px; border-left: 4px solid #ffc107;">
                <strong>⚠️ {file}</strong><br>
                <small>File too large ({size_mb:.1f} MB) for web download. Find it in your <code>youtube_downloads</code> folder instead.</small>
            </div>
            '''
            print("⚠️  Skipped (too large)")
            continue
        
        try:
            with open(filepath, "rb") as f:
                b64 = base64.b64encode(f.read()).decode()
            
            # Determine MIME type
            ext = os.path.splitext(file)[1].lower()
            mime_types = {
                '.mp4': 'video/mp4',
                '.mp3': 'audio/mpeg',
                '.webm': 'video/webm',
                '.m4a': 'audio/mp4',
            }
            mime_type = mime_types.get(ext, 'application/octet-stream')
            
            html_output += f'''
            <div style="background: white; padding: 15px; margin: 10px 0; border-radius: 5px; border: 1px solid #ddd;">
                <strong>📄 {file}</strong><br>
                <small style="color: #666;">Size: {size_mb:.2f} MB</small><br>
                <a href="data:{mime_type};base64,{b64}" 
                   download="{file}" 
                   style="display: inline-block; margin-top: 10px; background: #1976d2; color: white; 
                          padding: 10px 20px; text-decoration: none; border-radius: 5px; font-weight: bold;">
                   ⬇️ Download
                </a>
            </div>
            '''
            file_found = True
            print("✅ Ready")
        except Exception as e:
            html_output += f'''
            <div style="background: #ffebee; padding: 15px; margin: 10px 0; border-radius: 5px;">
                <strong>❌ Error with {file}</strong><br>
                <small>{str(e)}</small>
            </div>
            '''
            print(f"❌ Error")
    
    if not file_found:
        html_output += '<p style="color: #666;">All files are too large for web download. Check your <code>youtube_downloads</code> folder manually.</p>'
    
    html_output += '</div>'
    
    print(f"\n{'='*60}")
    print("✅ Download links generated!")
    print("='*60}\n")
    
    display(HTML(html_output))

---

## Step 7: Alternative - Save to Google Drive (For Google Colab)

**⚠️ This method only works in Google Colab, NOT in VS Code!**

If you're using VS Code:
- Use **Step 6** above for clickable download links
- Or check the `youtube_downloads` folder in your VS Code file explorer

If you're using Google Colab and want to save to Google Drive, run this cell:

In [ ]:
import shutil
from google.colab import drive

print("🔗 Mounting Google Drive...")
drive.mount('/content/drive')

# Define destination in Google Drive
drive_folder = '/content/drive/My Drive/YouTube_Downloads'

# Create folder if it doesn't exist
if not os.path.exists(drive_folder):
    os.makedirs(drive_folder)

print(f"\n📂 Copying files to Google Drive...\n")

copied_count = 0
for root, dirs, files in os.walk(DOWNLOAD_FOLDER):
    for file in files:
        source = os.path.join(root, file)
        destination = os.path.join(drive_folder, file)
        
        try:
            shutil.copy(source, destination)
            size_mb = os.path.getsize(source) / (1024 * 1024)
            print(f"✅ Copied: {file} ({size_mb:.2f} MB)")
            copied_count += 1
        except Exception as e:
            print(f"❌ Error copying {file}: {e}")

print(f"\n🎉 Complete! {copied_count} files copied to Google Drive.")
print(f"📁 Location: My Drive/YouTube_Downloads/")
print("\n🌐 Visit https://drive.google.com to access your files.")

---

## 📝 Tips and Notes

### Troubleshooting:
- **Error downloading:** The video might be restricted or unavailable in your region
- **Slow downloads:** YouTube might be throttling. Try again later
- **Audio download fails:** Make sure FFmpeg is installed (usually included with yt-dlp)

### Legal Notice:
- ⚠️ Only download videos you have permission to download
- Respect copyright laws and YouTube's Terms of Service
- This tool is for educational purposes and personal use only

### Supported Sites:
Besides YouTube, `yt-dlp` supports 1000+ sites including:
- Vimeo, Dailymotion, Facebook, Instagram
- Twitter, TikTok, Reddit
- And many more!

Just paste the video URL and it will work! 🎉